In [ ]:
import jax
import jax.numpy as jnp
import optax
import omegaconf
import hydra
from e3response.data import qm9_nmr
import reax

In [ ]:
# === CONFIG ===
CONFIG_PATH = "/home/mattia/Desktop/MLcodes/e3response/logs/train/runs/nequip-nmr_100mol/config.yaml"
CKPT_PATH = "/home/mattia/Desktop/MLcodes/e3response/logs/train/runs/nequip-nmr_100mol/checkpoints/last.ckpt"
ATOM_INDEX = 4  # <-- atomo da ottimizzare
STEPS = 200
LR = 1e-2
NOISE_STD = 0.1
MIN_DIST = 0.8  # distanza minima ammessa tra atomi

In [ ]:
# === LOAD MODEL ===
cfg = omegaconf.OmegaConf.load(CONFIG_PATH)
module: reax.Module = hydra.utils.instantiate(cfg.model, _convert_="object")
ckpt = reax.training.get_default_checkpointing().load(CKPT_PATH)
module.set_parameters(ckpt["parameters"])

In [ ]:
# === LOAD MOLECULE ===
dataset = qm9_nmr.QM9NMR(split="test")
example = dataset[0]  # oppure scegli l'indice che vuoi

positions = example["nodes"]["positions"]
species = example["nodes"]["species"]
target_tensor = example["nodes"]["NMR_tensors"][ATOM_INDEX]

In [ ]:
# === ADD NOISE TO SINGLE ATOM ===
key = jax.random.PRNGKey(0)
noise = jax.random.normal(key, shape=(3,)) * NOISE_STD
noisy_atom_pos = positions[ATOM_INDEX] + noise
original_positions = positions
noisy_positions = original_positions.at[ATOM_INDEX].set(noisy_atom_pos)

# === MIN DIST PENALTY ===
def min_distance_penalty(positions, min_dist=MIN_DIST):
    diffs = positions[:, None, :] - positions[None, :, :]
    dists = jnp.linalg.norm(diffs, axis=-1)
    mask = jnp.triu(jnp.ones_like(dists), k=1)
    penalty = jnp.where((dists < min_dist) & (mask == 1), (min_dist - dists) ** 2, 0.0)
    return jnp.sum(penalty)

# === LOSS FN ===
def loss_fn(atom_pos, fixed_positions, species, target_tensor, atom_index):
    full_positions = fixed_positions.at[atom_index].set(atom_pos)
    graph = module.build_graph_from_positions(positions=full_positions, atomic_numbers=species)
    pred = module(graph)
    predicted_tensor = pred["nodes"]["NMR_tensors_predicted"][atom_index]
    tensor_loss = jnp.mean((predicted_tensor - target_tensor) ** 2)
    penalty = min_distance_penalty(full_positions)
    return tensor_loss + 10.0 * penalty

# === OPTIMIZATION LOOP ===
def optimize_atom_position(atom_index, initial_positions, species, target_tensor, steps=STEPS, lr=LR):
    atom_pos = initial_positions[atom_index]
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(atom_pos)

    for step in range(steps):
        loss, grad = jax.value_and_grad(loss_fn)(atom_pos, initial_positions, species, target_tensor, atom_index)
        updates, opt_state = optimizer.update(grad, opt_state)
        atom_pos = optax.apply_updates(atom_pos, updates)

        if step % 10 == 0:
            print(f"[{step}] Loss: {loss:.6f}")

    optimized_positions = initial_positions.at[atom_index].set(atom_pos)
    return optimized_positions

# === RUN OPTIMIZATION ===
optimized_positions = optimize_atom_position(
    ATOM_INDEX,
    noisy_positions,
    species,
    target_tensor,
)

# === OPTIONAL: Compare before/after ===
import numpy as np
print("\nOriginal Position:", np.round(original_positions[ATOM_INDEX], 4))
print("Noisy Position   :", np.round(noisy_positions[ATOM_INDEX], 4))
print("Optimized Position:", np.round(optimized_positions[ATOM_INDEX], 4))
